# 📚 LangChain সিরিজ — Notebook 010

## Embeddings ও Vector Store

এই notebook-এ আমরা শিখব:

- **Embedding** কী এবং কীভাবে text থেকে vector তৈরি হয়
- **HuggingFace** দিয়ে locally (free) embedding করা
- **FAISS** দিয়ে vector store তৈরি, সংরক্ষণ ও search করা
- **Retriever** দিয়ে LangChain chain-এর সাথে জুড়ে RAG pipeline বানানো


# 🔢 Embeddings ও Vector Store — বিস্তারিত

এই অংশে আমরা দেখব Embedding আসলে কী, কীভাবে text থেকে vector তৈরি হয়,
এবং সেই vector গুলো কীভাবে FAISS-এ সংরক্ষণ করে similarity search করা যায়।


## কীভাবে কাজ করে?

পুরো প্রক্রিয়াটি ৪টি ধাপে হয়:

1. **Embed** — text-কে সংখ্যার তালিকায় (vector) রূপান্তর করো
2. **Store** — সেই vector গুলো একটি vector database-এ সংরক্ষণ করো
3. **Query** — ব্যবহারকারীর প্রশ্নকেও vector-এ রূপান্তর করো, তারপর সবচেয়ে কাছের vector খোঁজো
4. **Return** — সবচেয়ে মিলে যাওয়া top-k chunk গুলো ফেরত দাও

## আমরা যা ব্যবহার করব (সব Free, কোনো API লাগবে না):

- **HuggingFace Embeddings** — তোমার নিজের মেশিনে locally চলে
- **FAISS** — Facebook-এর তৈরি অত্যন্ত দ্রুত vector similarity search library

## সহজ ধারণা:

```
"বিড়াল"  →  [0.23, -0.41, 0.87 ...]  ─┐
                                         ├─ কাছাকাছি vector = কাছাকাছি অর্থ
"ছানা"   →  [0.21, -0.39, 0.85 ...]  ─┘

"গাড়ি"  →  [-0.91, 0.12, -0.34 ...]    ← দূরের vector = ভিন্ন অর্থ
```

## ধাপে ধাপে প্রক্রিয়া:

```
Text → Vector → Vector Database-এ store
প্রশ্ন → Vector → সবচেয়ে কাছের vector খোঁজো → সেই chunk ফেরত দাও
```


## 🤖 all-MiniLM-L6-v2 — Embedding Model পরিচিতি

আমরা যে embedding model ব্যবহার করব সেটি হলো `all-MiniLM-L6-v2`।  
এটি HuggingFace-এর সবচেয়ে জনপ্রিয় sentence embedding model গুলোর একটি।

| বৈশিষ্ট্য | বিবরণ |
|-----------|-------|
| ধরন | Sentence Transformer / Embedding Model |
| Parameters | ~২২ মিলিয়ন |
| Embedding আকার | **৩৮৪ dimension** (প্রতিটি text → ৩৮৪টি সংখ্যা) |
| Architecture | Transformer (MiniLM), ৬টি layer |
| Similarity পদ্ধতি | Cosine Similarity |
| Training Data | ১ বিলিয়নেরও বেশি sentence pair |
| সর্বোচ্চ input | ~২৫৬ token |

### কোথায় ব্যবহার হয়:
- Semantic Search (অর্থ বুঝে খোঁজা)
- Text Similarity (দুটি text কতটা কাছাকাছি)
- RAG সিস্টেম
- Clustering ও Retrieval

### কেন এই model?
✅ সম্পূর্ণ বিনামূল্যে  
✅ CPU-তেই চলে (GPU লাগে না)  
✅ হালকা ও দ্রুত  
✅ বাংলা সহ অনেক ভাষায় কাজ করে  


### ⚙️ Embedding Model লোড করা

`HuggingFaceEmbeddings` দিয়ে `all-MiniLM-L6-v2` model লোড করা হচ্ছে।  
প্রথমবার চালালে model download হবে — পরের বার local cache থেকে লোড হবে।

> 💡 **Install:** `pip install langchain-huggingface sentence-transformers`


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# HuggingFace embedding model লোড করো (সম্পূর্ণ বিনামূল্যে, locally চলে)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embedding model লোড হয়েছে!")
print(f"Model: all-MiniLM-L6-v2")


### 🔍 একটি Text কে Vector-এ রূপান্তর করা

`embeddings.embed_query()` দিয়ে যেকোনো text-কে vector-এ রূপান্তর করা যায়।  
output হলো ৩৮৪টি সংখ্যার একটি list — এটাই ওই text-এর "অর্থের প্রতিনিধিত্ব"।

```
"LangChain is a framework..."  →  [0.045, -0.123, 0.891, ... ]  (৩৮৪টি সংখ্যা)
```


In [ ]:
# একটি text-কে vector-এ রূপান্তর করো
sample_text = "LangChain is a framework for building LLM applications."

vector = embeddings.embed_query(sample_text)

print(f"Text: {sample_text}")
print(f"Vector dimension: {len(vector)}")
print(f"প্রথম ৫টি মান: {[round(v, 4) for v in vector[:5]]}")


---

## 🧪 Similarity — কাছাকাছি অর্থের text কি কাছাকাছি vector দেয়?

এখানে আমরা ৩টি text-এর vector তৈরি করব এবং দেখব কোনগুলো একে অপরের কাছাকাছি।  
**Cosine Similarity** ব্যবহার করা হবে — মান ১ মানে একদম একই, ০ মানে সম্পূর্ণ আলাদা।


In [ ]:
import numpy as np

# ৩টি ভিন্ন অর্থের text
texts = [
    "LangChain is a framework for building LLM applications.",
    "LangChain helps developers create AI-powered apps.",  # প্রথমটির কাছাকাছি
    "Python is a popular programming language."             # সম্পূর্ণ আলাদা
]

vectors = embeddings.embed_documents(texts)

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_1_2 = cosine_similarity(vectors[0], vectors[1])
sim_1_3 = cosine_similarity(vectors[0], vectors[2])

print(f"Text 1 vs Text 2 (কাছাকাছি অর্থ): {sim_1_2:.4f}")
print(f"Text 1 vs Text 3 (ভিন্ন অর্থ)   : {sim_1_3:.4f}")
print()
print("→ বেশি similarity মানে অর্থ কাছাকাছি!")


---

## 🗄️ FAISS Vector Store — Vector সংরক্ষণ ও দ্রুত খোঁজা

হাজার হাজার vector-এর মধ্যে সবচেয়ে কাছেরটা খুঁজে বের করা সহজ কাজ না।  
**FAISS** (Facebook AI Similarity Search) এই কাজটি অত্যন্ত দ্রুত করতে পারে।

### কীভাবে কাজ করে:

```
Document Chunks
      ↓  embed_documents()
   Vectors
      ↓  FAISS.from_documents()
  FAISS Index (disk বা memory-তে)
      ↓  similarity_search(query)
  সবচেয়ে কাছের top-k chunks
```

> 💡 **Install:** `pip install faiss-cpu`


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# কিছু sample document তৈরি করি
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models.",
             metadata={"source": "langchain_docs", "page": 1}),
    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.",
             metadata={"source": "faiss_docs", "page": 1}),
    Document(page_content="HuggingFace provides thousands of pretrained models for NLP tasks.",
             metadata={"source": "huggingface_docs", "page": 1}),
    Document(page_content="Vector databases store embeddings and allow fast nearest-neighbor search.",
             metadata={"source": "vector_db_guide", "page": 1}),
    Document(page_content="RAG combines retrieval with generation to answer questions from documents.",
             metadata={"source": "rag_guide", "page": 1}),
    Document(page_content="Embeddings convert text into numerical vectors that capture semantic meaning.",
             metadata={"source": "embedding_guide", "page": 1}),
]

# FAISS vector store তৈরি করো
vector_store = FAISS.from_documents(documents, embeddings)

print(f"✅ FAISS Vector Store তৈরি হয়েছে!")
print(f"মোট documents: {len(documents)}")


---

## 🔎 Similarity Search — প্রশ্নের সাথে মিলে যাওয়া Chunk খোঁজা

`similarity_search()` দিয়ে যেকোনো প্রশ্নের সবচেয়ে কাছের document গুলো বের করা যায়।  
ভেতরে কী হয়:
1. প্রশ্নটিকে vector-এ রূপান্তর করা হয়
2. FAISS সব stored vector-এর সাথে তুলনা করে
3. সবচেয়ে কাছের `k`টি document ফেরত দেয়


In [ ]:
# প্রশ্ন দিয়ে similarity search
query = "What is LangChain used for?"

results = vector_store.similarity_search(query, k=3)

print(f"প্রশ্ন: {query}")
print(f"সবচেয়ে কাছের {len(results)}টি Document:\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Content : {doc.page_content}")
    print(f"Source  : {doc.metadata['source']}")
    print()


### 📊 Score সহ Similarity Search

`similarity_search_with_score()` দিয়ে প্রতিটি result-এর সাথে similarity score পাওয়া যায়।  
FAISS-এ score হলো **L2 distance** — মান যত **কম**, তত বেশি মিল।


In [ ]:
query = "How do vector databases work?"

results_with_score = vector_store.similarity_search_with_score(query, k=3)

print(f"প্রশ্ন: {query}\n")

for i, (doc, score) in enumerate(results_with_score):
    print(f"--- Result {i+1} ---")
    print(f"Score   : {score:.4f}  (কম মানে বেশি মিল)")
    print(f"Content : {doc.page_content}")
    print()


---

## 💾 FAISS Vector Store সংরক্ষণ ও পুনরায় লোড করা

প্রতিবার program চালালে নতুন করে embed করতে হলে সময় ও খরচ দুটোই বেশি লাগে।  
তাই FAISS index disk-এ save করে রাখা যায় এবং পরে সরাসরি load করা যায়।

```
প্রথমবার: embed করো → FAISS-এ store করো → disk-এ save করো
পরের বার: disk থেকে load করো → সরাসরি search করো  ✅ অনেক দ্রুত
```


In [ ]:
import os

# Vector store disk-এ save করো
save_path = "faiss_index"
vector_store.save_local(save_path)
print(f"✅ FAISS index সংরক্ষিত হয়েছে: '{save_path}/' ফোল্ডারে")
print(f"ফাইলগুলো: {os.listdir(save_path)}")


In [ ]:
# পরে যেকোনো সময় load করো — আর embed করতে হবে না
loaded_vector_store = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True  # local file হলে safe
)

# load করা store দিয়ে search করো
test_results = loaded_vector_store.similarity_search("What is RAG?", k=2)
print("✅ Loaded FAISS থেকে search সফল!")
for doc in test_results:
    print(f"→ {doc.page_content}")


---

## 🔗 Retriever — Vector Store কে LangChain Chain-এ যুক্ত করা

Vector Store-কে সরাসরি LangChain chain-এ ব্যবহার করতে হলে `retriever`-এ রূপান্তর করতে হয়।  
`as_retriever()` দিয়ে এটি করা হয় — তারপর chain-এর সাথে `|` দিয়ে জুড়ে দেওয়া যায়।

```
User Question
      ↓
  Retriever  (vector store থেকে সবচেয়ে কাছের chunks বের করে)
      ↓
  LLM  (সেই chunks context হিসেবে পেয়ে উত্তর দেয়)
      ↓
  Final Answer
```


In [ ]:
# Vector Store → Retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # সবচেয়ে কাছের ৩টি chunk আনবে
)

# Retriever দিয়ে search
query = "How does RAG work?"
retrieved_docs = retriever.invoke(query)

print(f"প্রশ্ন: {query}")
print(f"Retrieved {len(retrieved_docs)}টি document:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"{i+1}. {doc.page_content}")


---

## 🚀 পূর্ণ RAG Pipeline — সব একসাথে

এখন সব কিছু একসাথে জুড়ে একটি পূর্ণ RAG chain তৈরি করা হচ্ছে।

```
প্রশ্ন → Retriever → প্রাসঙ্গিক Chunks → LLM Prompt → উত্তর
```

> ⚠️ এই অংশ চালাতে হলে `GOOGLE_API_KEY` দরকার।  
> আগের notebook-এ যে Gemini model setup করা হয়েছিল সেটা এখানে ব্যবহার হচ্ছে।


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY
)

# RAG Prompt — context ও question দুটোই template-এ
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """তুমি একজন সহায়ক assistant। নিচের context ব্যবহার করে প্রশ্নের উত্তর দাও।
Context:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# পূর্ণ RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# প্রশ্ন করো
question = "What is LangChain and how does it help with RAG?"
answer = rag_chain.invoke(question)

print(f"প্রশ্ন: {question}")
print(f"\nউত্তর:\n{answer}")


---

> ✅ **মূল কথা:**
> - **Embedding** হলো text → vector রূপান্তর। কাছাকাছি অর্থ মানে কাছাকাছি vector।
> - **FAISS** হলো সেই vector গুলো store করে দ্রুত খোঁজার tool।
> - **Retriever** হলো FAISS-এর wrapper যা LangChain chain-এর সাথে কাজ করে।
> - **RAG** = Retriever + LLM — প্রশ্নের উত্তর দিতে document থেকে তথ্য এনে LLM-কে দেওয়া।

### পরের ধাপ:
এই জ্ঞান ব্যবহার করে যেকোনো PDF, ওয়েবসাইট বা ডকুমেন্ট থেকে প্রশ্নের উত্তর দেওয়ার  
পূর্ণ RAG chatbot তৈরি করা যাবে! 🎉
